## 대구

In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. 전처리 및 Anomaly가 적용된 데이터 불러오기
df = pd.read_csv('../data/polar_weather_preprocessed.csv', parse_dates=['Date'])

# 2. 분석 타겟 설정: 대구 여름 이상 폭염 (장보고기지 래그 14일 적용)
daegu_lag = 14

df['Jangbogo_Lag_D'] = df['Jangbogo_Temp_Mean_Anomaly'].shift(daegu_lag)
df['Jangbogo_Lag_D_MA3'] = df['Jangbogo_Temp_Mean_Anomaly'].shift(daegu_lag).rolling(window=3).mean()
df['Sejong_Lag_D'] = df['Sejong_Temp_Mean_Anomaly'].shift(daegu_lag)
df['Month'] = df['Date'].dt.month

# 3. 정답지(Label) 생성: 평년 대비 불쾌지수가 비정상적으로 높은 상위 10% 날짜
threshold_thi_d = df['Daegu_THI_Max_Anomaly'].quantile(0.90)
df['Is_Heatwave_Daegu'] = (df['Daegu_THI_Max_Anomaly'] >= threshold_thi_d).astype(int)

# 4. 결측치 제거 및 데이터 분리
features_d = ['Jangbogo_Lag_D', 'Jangbogo_Lag_D_MA3', 'Sejong_Lag_D', 'Month']
ml_data_d = df[features_d + ['Is_Heatwave_Daegu']].dropna()

X_d = ml_data_d[features_d]
y_d = ml_data_d['Is_Heatwave_Daegu']

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(X_d, y_d, test_size=0.2, random_state=42)

# 5. 모델 학습 (클래스 가중치 적용)
model_d = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
model_d.fit(X_train_d, y_train_d)

# 6. 예측 및 성능 평가
y_pred_d = model_d.predict(X_test_d)

print("=== 🚀 [대구] 이상 기후(Anomaly) 기반 폭염 예측 모델 성능 ===")
print(f"✅ 모델 정확도(Accuracy): {accuracy_score(y_test_d, y_pred_d) * 100:.2f}%\n")
print("📊 상세 분류 리포트:")
print(classification_report(y_test_d, y_pred_d))

print("-" * 40)
print("🔍 변수 중요도")
for name, imp in zip(features_d, model_d.feature_importances_):
    print(f"🔹 {name}: {imp*100:.1f}%")

=== 🚀 [대구] 이상 기후(Anomaly) 기반 폭염 예측 모델 성능 ===
✅ 모델 정확도(Accuracy): 89.84%

📊 상세 분류 리포트:
              precision    recall  f1-score   support

           0       0.91      0.99      0.95       853
           1       0.36      0.05      0.09        92

    accuracy                           0.90       945
   macro avg       0.63      0.52      0.52       945
weighted avg       0.85      0.90      0.86       945

----------------------------------------
🔍 변수 중요도
🔹 Jangbogo_Lag_D: 26.8%
🔹 Jangbogo_Lag_D_MA3: 27.8%
🔹 Sejong_Lag_D: 30.3%
🔹 Month: 15.0%


## 부산

In [15]:
# 1. 부산 분석 타겟 설정
busan_lag = 46

df['Jangbogo_Lag_B'] = df['Jangbogo_Temp_Mean_Anomaly'].shift(busan_lag)
df['Jangbogo_Lag_B_MA3'] = df['Jangbogo_Temp_Mean_Anomaly'].shift(busan_lag).rolling(window=3).mean()
df['Sejong_Lag_B'] = df['Sejong_Temp_Mean_Anomaly'].shift(busan_lag)

# 2. 정답지(Label) 생성: 부산 이상 폭염 (상위 10%)
threshold_thi_b = df['Busan_THI_Max_Anomaly'].quantile(0.90)
df['Is_Heatwave_Busan'] = (df['Busan_THI_Max_Anomaly'] >= threshold_thi_b).astype(int)

# 3. 결측치 제거 및 데이터 분리
features_b = ['Jangbogo_Lag_B', 'Jangbogo_Lag_B_MA3', 'Sejong_Lag_B', 'Month']
ml_data_b = df[features_b + ['Is_Heatwave_Busan']].dropna()

X_b = ml_data_b[features_b]
y_b = ml_data_b['Is_Heatwave_Busan']

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X_b, y_b, test_size=0.2, random_state=42)

# 4. 모델 학습 (클래스 가중치 적용)
model_b = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
model_b.fit(X_train_b, y_train_b)

# 5. 예측 및 성능 평가
y_pred_b = model_b.predict(X_test_b)

print(f"=== 🌊 [부산] 이상 기후(Anomaly) 기반 폭염 예측 모델 성능 (Lag {busan_lag}일) ===")
print(f"✅ 모델 정확도(Accuracy): {accuracy_score(y_test_b, y_pred_b) * 100:.2f}%\n")
print("📊 상세 분류 리포트:")
print(classification_report(y_test_b, y_pred_b))

print("-" * 40)
print("🔍 변수 중요도")
for name, imp in zip(features_b, model_b.feature_importances_):
    print(f"🔹 {name}: {imp*100:.1f}%")

=== 🌊 [부산] 이상 기후(Anomaly) 기반 폭염 예측 모델 성능 (Lag 46일) ===
✅ 모델 정확도(Accuracy): 89.88%

📊 상세 분류 리포트:
              precision    recall  f1-score   support

           0       0.90      1.00      0.95       837
           1       0.73      0.11      0.19       102

    accuracy                           0.90       939
   macro avg       0.82      0.55      0.57       939
weighted avg       0.88      0.90      0.86       939

----------------------------------------
🔍 변수 중요도
🔹 Jangbogo_Lag_B: 26.6%
🔹 Jangbogo_Lag_B_MA3: 27.7%
🔹 Sejong_Lag_B: 29.9%
🔹 Month: 15.8%
